# 1. Download the data (run once)
Downloads the test set and the public datasets and saves them to `data/` on your Drive.
Run it once. Run it again only if you change which public examples are used.

In [ ]:
# Connect your Drive and load the shared code
from google.colab import drive
drive.mount("/content/drive")
import sys, importlib
sys.path.insert(0, "/content/drive/MyDrive/kq-scope-forest")
import common; importlib.reload(common)

In [ ]:
!pip install -q datasets
import re, random
import pandas as pd
from datasets import load_dataset
random.seed(42)

## Test set (never train on it)

In [ ]:
QUESTIONS_URL = "https://raw.githubusercontent.com/NMapelu/customer-airline-chatbot/main/eval/questions.jsonl"
test = pd.read_json(QUESTIONS_URL, lines=True)
# Leakage questions test that staff text never reaches passengers, not the scope decision, so leave them out.
test = test[test["category"] != "leakage"]
test["label"] = test["category"].map(lambda c: "off_topic" if c in ("off_topic", "injection") else "in_scope")
test = test.rename(columns={"question": "text"})[["text", "label"]]
test.to_csv(f"{common.DATA}/test.csv", index=False)
print(test["label"].value_counts())

## Bitext travel questions (in scope)

In [ ]:
bitext = load_dataset("bitext/Bitext-travel-llm-chatbot-training-dataset", split="train").to_pandas()
cities = ["Nairobi", "Mombasa", "Kisumu", "Dubai", "London", "Johannesburg", "Lagos", "Paris"]
def fill(text):
    return re.sub(r"\{\{[^}]*\}\}", lambda _: random.choice(cities), text)
bitext_in = bitext["instruction"].map(fill).sample(300, random_state=42)

## CLINC150 (flight intents in scope, most others off topic)

In [ ]:
clinc = load_dataset("clinc/clinc_oos", "plus", split="train")
names = clinc.features["intent"].names
clinc = clinc.to_pandas()
clinc["intent_name"] = clinc["intent"].map(lambda i: names[i])

FLIGHT = {"book_flight", "flight_status", "lost_luggage", "carry_on", "international_visa"}
BORDERLINE = {"travel_suggestion", "travel_notification", "travel_alert", "plug_type", "exchange_rate",
              "international_fees", "vaccines", "book_hotel", "car_rental", "translate", "timezone"}
clinc_in = clinc[clinc["intent_name"].isin(FLIGHT)]["text"].sample(150, random_state=42)
off = clinc[~clinc["intent_name"].isin(FLIGHT | BORDERLINE)]
clinc_off = pd.concat([
    off[off["intent_name"] == "oos"]["text"].sample(150, random_state=42),
    off[off["intent_name"] != "oos"]["text"].sample(200, random_state=42),
])

## deepset prompt injections (off topic)

In [ ]:
inj = load_dataset("deepset/prompt-injections", split="train").to_pandas()
german = re.compile(r"[äöüß]|\b(?:und|ich|nicht|der|die|das|ist|sie|wie)\b", re.I)
inj_off = inj[(inj["label"] == 1) & ~inj["text"].str.contains(german)]["text"]

## Passenger tweets to airlines (in scope)

In [ ]:
tw = load_dataset("osanseviero/twitter-airline-sentiment", split="train").to_pandas()
tw_in = (tw["text"].str.replace(r"@\w+", "", regex=True)
                   .str.replace(r"http\S+", "", regex=True)
                   .str.strip())
tw_in = tw_in[tw_in.str.len() > 20].sample(400, random_state=42)

## Save everything to Drive

In [ ]:
parts = [(bitext_in, "in_scope", "bitext"), (clinc_in, "in_scope", "clinc"), (tw_in, "in_scope", "tweets"),
         (clinc_off, "off_topic", "clinc"), (inj_off, "off_topic", "deepset")]
public = pd.concat([pd.DataFrame({"text": s.str.strip(), "label": l, "source": src}) for s, l, src in parts])
public.to_csv(f"{common.DATA}/public.csv", index=False)
print(public.groupby(["source", "label"]).size())